# KPI APP 


we will test a UI that helps us clculate KPI for each artifact 


## Imports


In [2]:
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 16.9 MB/s eta 0:00:00


In [3]:
# --- Notebook environment setup ---
from pathlib import Path
import sys
import pandas as pd

# Make the src/ folder importable
PROJECT_ROOT = Path("..").resolve()        # notebook in notebooks/
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# --- Import the reusable app layer ---
from batterydata.apps.artifacts import ArtifactStore
from batterydata.apps.kpi_registry import list_presets, list_all, resolve_kpis, get_preset
from batterydata.apps.selectors import pretty_print_blocks, block_ids_from_doe, build_signature_map, filter_block_ids_by_signature_regex
from batterydata.apps.kpi_runner import run_kpis_for_blocks, simple_sanity_checks

# Widgets
import ipywidgets as w
from IPython.display import display, clear_output


In [ ]:
from pathlib import Path
from batterydata.kpi_jobs.config_builder import create_job_file

PROJECT_ROOT = Path(".").resolve()
job_path = create_job_file(
    project_root=PROJECT_ROOT,
    cell="CellA",
    blocks=[2],
    cycles="all",              # or {"range":"1-200","every":1} or [1,2,3]
    kpi_preset="cycle_core",   # or kpis=["QChargeAh", "QDischargeAh", ...]
    strict_single_block=True,
    notes="First KPI run for block 2",
)
print("Job created at:", job_path)


## Set up and Discover artifacts 

## Block Selection UI

In [ ]:
out_block_table = w.Output()

# Build dynamic block controls (will update when cell changes)
txt_regex = w.Text(placeholder="e.g. CH_.*_CV", description="Regex:")
btn_apply_regex = w.Button(description="Filter", icon="filter", button_style="")
sel_blocks = w.SelectMultiple(options=[], description="Block IDs:", rows=8, layout=w.Layout(width="240px"))
btn_select_all = w.Button(description="Select all", icon="check")
btn_clear = w.Button(description="Clear", icon="times")

def update_block_controls(cell_dir):
    manifest = store.load_manifest(cell_dir)
    doe = store.load_table_from_manifest(manifest, "doe_table")
    blocks = store.load_blocks(cell_dir)
    bids = block_ids_from_doe(doe)
    sel_blocks.options = bids
    sel_blocks.value = tuple(bids)  # preselect all by default

    # show id→signature map (best-effort)
    sig_map = build_signature_map(blocks) if blocks else {}
    with out_block_table:
        clear_output()
        if sig_map:
            display(pd.DataFrame(
                [{"block_id": k, "signature": v} for k, v in sig_map.items()]
            ))
        else:
            print("No signature map available (blocks.json missing or empty).")

def on_apply_regex_clicked(_):
    cell_dir = dd_cell.value
    manifest = store.load_manifest(cell_dir)
    doe = store.load_table_from_manifest(manifest, "doe_table")
    blocks = store.load_blocks(cell_dir)
    bids = block_ids_from_doe(doe)
    sig_map = build_signature_map(blocks) if blocks else {}
    pattern = txt_regex.value.strip()
    if pattern and sig_map:
        filtered = filter_block_ids_by_signature_regex(sig_map, pattern)
        filtered = [b for b in filtered if b in bids]  # intersect with actual DoE IDs
        sel_blocks.options = filtered
        sel_blocks.value = tuple(filtered)
    elif not pattern:
        sel_blocks.options = bids
        sel_blocks.value = tuple(bids)

def on_select_all_clicked(_):
    sel_blocks.value = tuple(sel_blocks.options)

def on_clear_clicked(_):
    sel_blocks.value = ()

btn_apply_regex.on_click(on_apply_regex_clicked)
btn_select_all.on_click(on_select_all_clicked)
btn_clear.on_click(on_clear_clicked)

# handle changing cell
def on_cell_change(change):
    if change["name"] == "value":
        update_block_controls(change["new"])

dd_cell.observe(on_cell_change, names="value")

display(w.HBox([txt_regex, btn_apply_regex, btn_select_all, btn_clear]))
display(w.VBox([w.Label("Block signatures (best-effort mapping):"), out_block_table]))
display(sel_blocks)

# initial populate
update_block_controls(dd_cell.value)
